# 4.1 Векторы и матрицы в NumPy: решения упражнений

Аналитика 101 · блок «Дата аналитика» · тема 04 «Математика для данных» · страница 4.1.

Сначала идут шесть блоков кода со страницы, решения с разбором — во второй половине. Ноутбук выполнен целиком сверху вниз на NumPy 2.5, все проверки прошли.

Загляните сюда после своей попытки: чужое решение понятнее, когда уже есть своё.

## Где лежат данные

Код страницы читает файлы по путям от корня папки с материалами: `data/clients/clients.csv` и `data/shop/shop.sqlite`. Ноутбук лежит в папке страницы, поэтому первая ячейка поднимается до папки, в которой есть `data/`, и делает её рабочей.

In [1]:
import os
import pathlib
import platform

import numpy as np
import pandas as pd

root = pathlib.Path.cwd()
while not (root / "data" / "clients").is_dir() and root != root.parent:
    root = root.parent                      # поднимаемся, пока рядом не окажется data/clients/
assert (root / "data" / "clients" / "clients.csv").exists(), \
    "Не найдена папка data/clients/. Скачайте общий набор темы 04 и положите data/ рядом с папкой страницы"
assert (root / "data" / "shop" / "shop.sqlite").exists(), \
    "Не найдена папка data/shop/. Скачайте общий набор темы 03 и положите его в ту же папку data/"
os.chdir(root)

print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

Python 3.13.18 | numpy 2.5.3 | pandas 3.0.5


## Код страницы

Блоки идут в том же порядке, что на странице: сначала три блока из раздела «Нюансы», затем три из раздела «Код». Каждый блок заново читает таблицу, поэтому их можно запускать по отдельности.

### (5,) — пять чисел, (5, 1) — столбец

In [2]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

age = X[:5, 0]                 # один индекс — ось пропадает
print(age, age.shape)
print(X[:5, [0]].shape, X[:5, 0:1].shape)   # список или срез — ось остаётся
col = age.reshape(-1, 1)       # то же, что age[:, None]
print(col.shape)
print(age.T.shape)             # у одномерного массива транспонировать нечего
print(col.T.shape)

[66 34 48 41 59] (5,)
(5, 1) (5, 1)
(5, 1)
(5,)
(1, 5)


Транспонирование одномерного массива возвращает тот же (5,): у него одна ось, менять местами нечего. Столбец получают reshape(-1, 1) или индексом None.

### (3,) + (3, 1) — таблица, (1000,) − (1000, 1) — миллион разностей

In [3]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

a = np.array([1, 2, 3])           # (3,)
b = np.array([[10], [20], [30]])  # (3, 1)
print(a + b)                      # (3,) встаёт строкой, (3, 1) — столбцом
print((a + b).shape)

X1 = np.c_[np.ones(len(X)), X[:, 1:]]
w = np.array([205, 0.0347, 2550])
spend = df["spend"].to_numpy()          # (1000,)
pred = (X1 @ w).reshape(-1, 1)          # (1000, 1): прогноз случайно стал столбцом
err = spend - pred                      # (1000,) − (1000, 1) → (1000, 1000), без ошибки
print(err.shape)
print(np.abs(err).mean().round())               # «средняя ошибка» по миллиону пар
print(np.abs(spend - pred.ravel()).mean().round())  # настоящая: (1000,) − (1000,)

[[11 12 13]
 [21 22 23]
 [31 32 33]]
(3, 3)
(1000, 1000)
7532.0
3017.0


Ошибки нет ни в одном случае: формы совместимы по правилу broadcasting. Настоящая средняя ошибка прогноза — 3 017 ₽, а по «матрице ошибок» 1 000 × 1 000 выходит 7 532 ₽.

### Целый dtype: дробь отбрасывается, на месте не вычесть, int32 переполняется

In [4]:
import numpy as np
import pandas as pd
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()
print(X.dtype)                     # все столбцы целые → int64

X[0, 1] = 80_500.7                 # дробь в целом массиве отбрасывается молча
print(X[0])

try:
    X -= X.mean(axis=0)            # на месте: результат float64 не влезает в int64
except TypeError as e:
    print(type(e).__name__, e)
Xc = X - X.mean(axis=0)            # новый массив: NumPy сам повышает до float64
print(Xc.dtype)

inc32 = X[:3, 1].astype(np.int32)  # «сэкономили память»
print(inc32 * inc32)               # 80 500² > 2 147 483 647: переполнение без предупреждения
print(X[:3, 1] * X[:3, 1])         # в int64 верно
print(X[:3, 1] // 1000, X[:3, 1] / 1000)   # // — целая часть, / — всегда float

int64
[   66 80500     2]
UFuncTypeError Cannot cast ufunc 'subtract' output from dtype('float64') to dtype('int64') with casting rule 'same_kind'
float64
[-2109684592   625000000   466032704]
[6480250000  625000000 4761000000]
[80 25 69] [80.5 25.  69. ]


UFuncTypeError — подкласс TypeError. В int32 помещаются числа до 2 147 483 647, а 80 500² = 6 480 250 000: переполнение в массиве прошло без ошибки и предупреждения.

### Таблица — матрица (1000, 3), строка — вектор (3,)

In [5]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

a = np.array([57, 118_000, 6])          # клиент 49: возраст, доход, заказы
print(a, a.shape, a.ndim, a.dtype)

df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()   # матрица «клиенты × признаки»
print(X.shape, X.dtype)

print(X[48])              # строка 48 — клиент 49 (счёт с нуля)
print(X[:3, 1])           # срез: доход первых трёх клиентов
print(X[:, 2].max())      # столбец «заказы» целиком, максимум
print(X[X[:, 2] >= 12])   # строки по условию: 12 заказов и больше

X1 = np.c_[np.ones(len(X)), X[:, 1:]]   # (1000, 3): единица, доход, заказы
w = np.array([205, 0.0347, 2550])        # веса трат: как их находят — на странице 4.2
pred = X1 @ w                            # скалярное произведение каждой строки на w
print(X1.shape, w.shape, pred.shape)
print(pred[:3], df["spend"].to_numpy()[:3])

[    57 118000      6] (3,) 1 int64
(1000, 3) int64
[    57 118000      6]
[80000 25000 69000]
15
[[    60 314000     15]
 [    44 273000     12]
 [    70 185000     12]]
(1000, 3) (3,) (1000,)
[ 8081.   8722.5 20449.3] [ 9080 10020 15900]


X[48] — строка клиента 49: счёт строк с нуля. X1 @ w — 1 000 скалярных произведений строки на w без цикла: у первых трёх клиентов прогноз 8 081, 8 722,5 и 20 449,3 ₽ при настоящих тратах 9 080, 10 020 и 15 900 ₽.

### Расстояния между всеми клиентами — без единого цикла

In [6]:
import numpy as np
import pandas as pd
np.set_printoptions(precision=2, suppress=True)
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy()

Z = (X - X.mean(axis=0)) / X.std(axis=0)   # стандартизация: (1000, 3) − (3,), потом / (3,)

S = Z[:6]                                  # первые 6 клиентов
diff = S[:, None, :] - S[None, :, :]       # (6, 1, 3) − (1, 6, 3) → (6, 6, 3)
D = np.sqrt((diff ** 2).sum(axis=-1))      # сумма по признакам → (6, 6)
print(diff.shape, D.shape)
print(D)

R = X[:6]                                  # те же клиенты без масштаба
print(np.sqrt(((R[:, None, :] - R[None, :, :]) ** 2).sum(axis=-1)).round())

# все 1000 × 1000 через скалярные произведения: |a − b|² = |a|² + |b|² − 2 a·b
sq = (Z ** 2).sum(axis=1)                            # (1000,)
D2 = sq[:, None] + sq[None, :] - 2 * Z @ Z.T         # (1000, 1000)
D_all = np.sqrt(np.clip(D2, 0, None))                # отрицательные −1e-15 — погрешность округления
near = np.argsort(D_all[48])[:4]                     # клиент 49 и трое ближайших
print(df.iloc[near, :4].assign(dist=D_all[48, near].round(3)).to_string(index=False))

(6, 6, 3) (6, 6)
[[0.   2.56 2.59 1.71 1.04 2.46]
 [2.56 0.   2.35 1.5  2.09 0.18]
 [2.59 2.35 0.   2.8  1.56 2.27]
 [1.71 1.5  2.8  0.   1.81 1.37]
 [1.04 2.09 1.56 1.81 0.   1.99]
 [2.46 0.18 2.27 1.37 1.99 0.  ]]
[[    0. 55000. 11000. 13000.  7000. 48000.]
 [55000.     0. 44000. 42000. 48000.  7000.]
 [11000. 44000.     0.  2000.  4000. 37000.]
 [13000. 42000.  2000.     0.  6000. 35000.]
 [ 7000. 48000.  4000.  6000.     0. 41000.]
 [48000.  7000. 37000. 35000. 41000.     0.]]
 client_id  age  income  orders  dist
        49   57  118000       6 0.000
       960   59  122000       6 0.166
       745   59  109000       6 0.268
       387   62  112000       6 0.358


Первая матрица — в стандартных единицах, вторая — в исходных, округлена до целых. Без масштаба ближе всех клиенты 3 и 4: доходы 69 000 и 67 000 ₽. В стандартных единицах это самая далёкая пара, 2,80: семь заказов против одного. clip нужен из-за погрешности округления: на месте нулей в D2 встречаются числа около −7·10⁻¹⁵, и корень из них дал бы nan.

### Косинус по покупкам: ближе всего товары того же бренда

In [7]:
import sqlite3
import numpy as np
import pandas as pd
np.set_printoptions(precision=3, suppress=True)

con = sqlite3.connect("data/shop/shop.sqlite")
long = pd.read_sql("""
    SELECT o.customer_id, p.product_name, SUM(oi.qty) AS qty
    FROM order_items AS oi
    JOIN orders   AS o USING (order_id)
    JOIN products AS p USING (product_id)
    WHERE o.status = 'paid'
    GROUP BY o.customer_id, p.product_name""", con)

M = long.pivot_table(index="customer_id", columns="product_name",
                     values="qty", fill_value=0)       # клиенты × товары
P = M.to_numpy(dtype=float)
print(P.shape)

a = P[:, M.columns.get_loc("Плед «Сова»")]             # столбец: кто сколько купил
b = P[:, M.columns.get_loc("Наволочка «Сова»")]
print(a @ b, np.linalg.norm(a).round(3), np.linalg.norm(b))
print(round(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)), 3))

Pn = P / np.linalg.norm(P, axis=0)                     # (51, 32) / (32,): столбцы длины 1
C = Pn.T @ Pn                                          # (32, 32): косинусы всех пар
i = M.columns.get_loc("Плед «Сова»")
top = np.argsort(-C[i])[1:4]                           # [0] — сам плед, cos = 1
print(pd.Series(C[i, top], index=M.columns[top]).round(3).to_string())
print((C[i] == 0).sum(), "товаров без единого общего покупателя: cos = 0")

(51, 32)
44.0 9.165 8.0
0.6


product_name
Свеча «Сова»        0.767
Чайник «Сова»       0.706
Наволочка «Сова»    0.600
10 товаров без единого общего покупателя: cos = 0


Оплаченные заказы есть у 51 клиента из 60, отсюда 51 строка. a @ b = 44 — сумма произведений штук по общим покупателям. В учебной базе заказ всегда из одного бренда, поэтому соседи пледа — товары «Совы». У 10 товаров с пледом нет ни одного общего покупателя: их векторы перпендикулярны, косинус ровно 0.

## Упражнения

Для упражнений берём ту же таблицу в вещественном типе и сразу стандартизуем её, как во втором блоке раздела «Код»: `Z` — клиенты в стандартных отклонениях от среднего.

In [8]:
np.set_printoptions(precision=3, suppress=True)
df = pd.read_csv("data/clients/clients.csv")
X = df[["age", "income", "orders"]].to_numpy(dtype=float)   # возраст, доход, заказы
Z = (X - X.mean(axis=0)) / X.std(axis=0)                    # те же клиенты в стандартных единицах
print(X.shape, X.dtype, Z.shape)
print(Z[48])                                                # клиент 49

(1000, 3) float64 (1000, 3)
[0.851 1.012 0.932]


### Упражнение 1. Какая ось сворачивается

Запишите в `col_mean` средние трёх признаков по всем клиентам, а в `row_max` — наибольшее из трёх чисел в строке каждого клиента. Таблица — `X`, без масштаба.

In [9]:
col_mean = X.mean(axis=0)     # axis=0 сворачивает строки: остаётся по числу на столбец
row_max = X.max(axis=1)       # axis=1 сворачивает столбцы: остаётся по числу на строку
print(col_mean, col_mean.shape)
print(row_max[:5], row_max.shape)

[   43.802 79323.        3.975] (3,)
[80000. 25000. 69000. 67000. 73000.] (1000,)


In [10]:
assert getattr(col_mean, "shape", None) == (3,), f"col_mean — по числу на признак, форма (3,); получено {getattr(col_mean, 'shape', None)}. Строки сворачивает axis=0"
assert np.allclose(col_mean, [43.802, 79323.0, 3.975]), f"Средние: возраст 43,802, доход 79 323, заказы 3,975; получено {col_mean}"
assert getattr(row_max, "shape", None) == (1000,), f"row_max — по числу на клиента, форма (1000,); получено {getattr(row_max, 'shape', None)}. Столбцы сворачивает axis=1"
assert np.allclose(row_max, X[:, 1]), "В каждой строке самое большое число — доход: row_max должен совпасть со столбцом X[:, 1]"
print("Упражнение 1: верно")

Упражнение 1: верно


**Разбор.** Номер оси говорит, какое измерение исчезнет из формы. У `X` форма (1000, 3): `axis=0` убирает первую длину, остаётся (3,), `axis=1` убирает вторую, остаётся (1000,). Наибольшее число в каждой строке — доход: в рублях он заслоняет остальные признаки, как на странице.

### Упражнение 2. Строки длины 1

Соберите матрицу `U`: каждая строка `Z`, делённая на свою длину. У всех строк `U` длина должна быть равна 1 — от клиента остаётся одно направление.

In [11]:
length = np.linalg.norm(Z, axis=1)      # длины строк, форма (1000,)
U = Z / length[:, None]                 # (1000, 3) / (1000, 1): столбец длин растягивается вправо
print(U.shape, np.linalg.norm(U, axis=1)[:4])

(1000, 3) [1. 1. 1. 1.]


In [12]:
assert getattr(U, "shape", None) == (1000, 3), f"U должна остаться матрицей (1000, 3); получено {getattr(U, 'shape', None)}"
assert np.allclose(np.linalg.norm(U, axis=1), 1), "Длина каждой строки U должна быть 1: разделите строку на её длину, а не столбец на его длину"
assert np.allclose(U[48], [0.526, 0.626, 0.576], atol=1e-3), f"Строка клиента 49 должна быть (0,526; 0,626; 0,576); получено {U[48]}"
print("Упражнение 2: верно")

Упражнение 2: верно


**Разбор.** Запись `Z / np.linalg.norm(Z, axis=1)` кончается ошибкой `ValueError: operands could not be broadcast together with shapes (1000,3) (1000,)`. Формы сравниваются справа: 3 и 1 000 не равны, и ни одна не единица. Длины надо поставить столбцом — `[:, None]`, `reshape(-1, 1)` или `np.linalg.norm(Z, axis=1, keepdims=True)`. Хуже было бы с квадратной таблицей (3, 3): там деление на (3,) прошло бы молча и разделило столбцы вместо строк.

### Упражнение 3. Расстояния от одного клиента до всех

Запишите в `d` расстояния от клиента 49 до каждого из 1 000 клиентов в стандартных единицах — без цикла. В `nearest` положите номера `client_id` трёх ближайших к нему клиентов, не считая его самого.

In [13]:
d = np.sqrt(((Z - Z[48]) ** 2).sum(axis=1))             # (1000, 3) − (3,), сумма по признакам
order = np.argsort(d)                                   # позиции по возрастанию расстояния
nearest = df["client_id"].to_numpy()[order[1:4]]        # order[0] — сам клиент 49
print(d.shape, d[order[:4]], nearest)

(1000,) [0.    0.166 0.268 0.358] [960 745 387]


In [14]:
assert getattr(d, "shape", None) == (1000,), f"d — по расстоянию на клиента, форма (1000,); получено {getattr(d, 'shape', None)}"
assert d[48] == 0, f"Расстояние от клиента 49 до самого себя — 0; получено {d[48]}"
assert np.allclose(d[[959, 744, 386]], [0.166, 0.268, 0.358], atol=1e-3), "До клиентов 960, 745 и 387 должно быть 0,166, 0,268 и 0,358. Считайте по Z, а не по X"
assert list(nearest) == [960, 745, 387], f"Ближайшие — клиенты 960, 745 и 387; получено {list(nearest)}. Строка 48 — это client_id 49"
print("Упражнение 3: верно")

Упражнение 3: верно


**Разбор.** `Z - Z[48]` вычитает вектор (3,) из каждой строки матрицы (1000, 3) — это первая строка таблицы broadcasting на странице. Массив (1000, 1000, 3) здесь не нужен: он понадобился бы для расстояний между всеми парами. Ответ совпал с выводом блока про расстояния. Если считать по `X`, ближайшими окажутся клиенты с тем же доходом, а возраст и заказы не повлияют.

### Упражнение 4. Косинус со всеми клиентами одной операцией

Запишите в `cos` косинус угла между стрелкой клиента 49 и стрелкой каждого клиента в стандартных единицах. В `same_side` — сколько клиентов смотрят в ту же сторону, что и клиент 49: у скольких косинус больше нуля.

In [15]:
a = Z[48]
cos = Z @ a / (np.linalg.norm(Z, axis=1) * np.linalg.norm(a))   # (1000, 3) @ (3,) → (1000,)
same_side = int((cos > 0).sum())
print(cos.shape, cos[[48, 880, 174, 988]], same_side)

(1000,) [ 1.    -0.928  0.002  0.904] 461


In [16]:
assert getattr(cos, "shape", None) == (1000,), f"cos — по числу на клиента, форма (1000,); получено {getattr(cos, 'shape', None)}"
assert np.isclose(cos[48], 1), f"Косинус клиента 49 с самим собой — 1; получено {cos[48]}"
assert np.isclose(cos[880], -0.928, atol=1e-3), f"С клиентом 881 косинус −0,928 (угол 158°); получено {cos[880]}. Поделили на обе длины?"
assert same_side == 461, f"В ту же сторону смотрит 461 клиент; получено {same_side}"
print("Упражнение 4: верно")

Упражнение 4: верно


**Разбор.** `Z @ a` — тысяча скалярных произведений, по одному на строку, как `X @ w` на странице. Делим на длину каждой строки и на длину `a`: обе величины формы (1000,) и число, поэтому новые оси не нужны. Знак косинуса совпадает со знаком скалярного произведения, отсюда 461 — то же число, что в последнем шаге анимации. С клиентами 881, 175 и 989 косинусы −0,928, 0,002 и 0,904: углы 158°, 90° и 25°.

### Упражнение 5. Ловушка с формой

Ниже прогноз трат `pred` случайно стал столбцом формы (1000, 1), как в блоке про broadcasting. Запишите в `bad_shape` форму массива `spend - pred`, а в `mae` — настоящую среднюю абсолютную ошибку прогноза в рублях.

In [17]:
X1 = np.c_[np.ones(len(df)), df["income"], df["orders"]]
w = np.array([205, 0.0347, 2550])
spend = df["spend"].to_numpy()             # (1000,)
pred = (X1 @ w).reshape(-1, 1)             # (1000, 1)
print(spend.shape, pred.shape)

(1000,) (1000, 1)


In [18]:
bad_shape = (spend - pred).shape                    # (1000,) − (1000, 1) → таблица всех пар
mae = np.abs(spend - pred.ravel()).mean()           # ravel возвращает прогнозу форму (1000,)
print(bad_shape, round(float(mae)))

(1000, 1000) 3017


In [19]:
assert bad_shape == (1000, 1000), f"(1000,) минус (1000, 1) даёт таблицу (1000, 1000); получено {bad_shape}"
assert isinstance(mae, (int, float, np.floating)) and round(float(mae)) == 3017, f"Настоящая ошибка — 3 017 ₽; получено {mae}. 7 532 ₽ — это среднее по миллиону чужих пар"
print("Упражнение 5: верно")

Упражнение 5: верно


**Разбор.** Вектор (1000,) встаёт строкой, столбец (1000, 1) — столбцом, и разность считается для каждой пары «траты клиента i, прогноз клиента j». Ошибки нет, а среднее по миллиону таких пар — 7 532 ₽ вместо 3 017 ₽. Лекарства два: вернуть прогнозу форму (1000,) через `ravel()` или сделать столбцом и траты. Привычка, которая ловит такие ошибки, — смотреть `.shape` после каждой операции с массивами разных форм.